# Regression Adjustment (Supplementary)

**This analysis is not part of the [analysis plan](../analysis_plan.md).** The main results remain the Welch estimates in [`01_01_average_effects.ipynb`](01_01_average_effects.ipynb).

**Question:** does adding pre-treatment customer features make the effect estimates more precise? In a randomized experiment, the groups are already balanced, so adjustment is not needed to remove bias. It can only narrow the confidence intervals, by explaining part of the variation in spend.

**Model, fixed on 2026-11-03 before running:**

`spend ~ mens_email + womens_email + recency + history + mens + womens + newbie + zip_code + channel`

- `mens_email` and `womens_email` are 0/1 indicators. The No E-Mail group is the baseline, with both set to 0.
- All other terms are pre-treatment features. `history_segment` is left out because it repeats the information in `history`.
- `visit` and `conversion` are left out because they happen after the email and could be affected by it.
- Standard errors are heteroskedasticity-robust (HC3), because most customers have zero spend.
- The model is run once and compared with the unadjusted estimates.

In [1]:
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import ttest_ind

df = pd.read_csv("../data/hillstrom.csv")
df["mens_email"] = (df["segment"] == "Mens E-Mail").astype(int)
df["womens_email"] = (df["segment"] == "Womens E-Mail").astype(int)

df.groupby("segment")[["mens_email", "womens_email"]].max()

,mens_email,womens_email
segment,,
Mens E-Mail,1,0
No E-Mail,0,0
Womens E-Mail,0,1


## Step 1: Fit the models

The unadjusted model contains only the two email indicators, so its coefficients are the plain differences in mean spend. The adjusted model adds the customer features.

In [2]:
features = "recency + history + mens + womens + newbie + C(zip_code) + C(channel)"
unadjusted = smf.ols("spend ~ mens_email + womens_email", df).fit(cov_type="HC3")
adjusted = smf.ols(f"spend ~ mens_email + womens_email + {features}", df).fit(cov_type="HC3")

print(f"R-squared: unadjusted {unadjusted.rsquared:.4f}, adjusted {adjusted.rsquared:.4f}")
adjusted.summary().tables[1]

R-squared: unadjusted 0.0004, adjusted 0.0014


,coef,std err,z,P>|z|,[0.025,0.975]
Intercept,0.3727,0.416,0.896,0.370,-0.442,1.188
C(zip_code)[T.Surburban],-0.1857,0.174,-1.065,0.287,-0.528,0.156
C(zip_code)[T.Urban],-0.1288,0.177,-0.729,0.466,-0.475,0.217
C(channel)[T.Phone],-0.0441,0.231,-0.191,0.849,-0.497,0.409
C(channel)[T.Web],0.1347,0.229,0.587,0.557,-0.315,0.584
mens_email,0.7664,0.145,5.281,0.000,0.482,1.051
womens_email,0.4258,0.130,3.268,0.001,0.170,0.681
recency,-0.0501,0.017,-2.895,0.004,-0.084,-0.016
history,0.0011,0.000,2.481,0.013,0.000,0.002
mens,0.6031,0.292,2.067,0.039,0.031,1.175


## Step 2: Compare the effect estimates

Each row compares one pair of groups. Mens vs Womens is the difference between the two email coefficients. The Welch column repeats the main results for reference.

In [3]:
contrasts = {
    "Mens vs No E-Mail": ("mens_email = 0", "Mens E-Mail", "No E-Mail"),
    "Womens vs No E-Mail": ("womens_email = 0", "Womens E-Mail", "No E-Mail"),
    "Mens vs Womens": ("mens_email - womens_email = 0", "Mens E-Mail", "Womens E-Mail"),
}

rows = []
for name, (hypothesis, treated, control) in contrasts.items():
    a = df.loc[df["segment"] == treated, "spend"]
    b = df.loc[df["segment"] == control, "spend"]
    welch_ci = ttest_ind(a, b, equal_var=False).confidence_interval(0.95)
    row = {"comparison": name, "welch_estimate": a.mean() - b.mean(), "welch_width": welch_ci.high - welch_ci.low}
    for label, model in [("unadjusted", unadjusted), ("adjusted", adjusted)]:
        test = model.t_test(hypothesis)
        low, high = test.conf_int(alpha=0.05)[0]
        row[f"{label}_estimate"] = float(test.effect[0])
        row[f"{label}_low"] = low
        row[f"{label}_high"] = high
        row[f"{label}_width"] = high - low
    rows.append(row)

comparison = pd.DataFrame(rows).set_index("comparison")
comparison["width_change"] = comparison["adjusted_width"] / comparison["unadjusted_width"] - 1
comparison.round(4)

,welch_estimate,welch_width,unadjusted_estimate,unadjusted_low,unadjusted_high,unadjusted_width,adjusted_estimate,adjusted_low,adjusted_high,adjusted_width,width_change
comparison,,,,,,,,,,,
Mens vs No E-Mail,0.7698,0.5694,0.7698,0.4851,1.0545,0.5694,0.7664,0.4820,1.0509,0.5689,-0.0009
Womens vs No E-Mail,0.4244,0.5109,0.4244,0.1690,0.6799,0.5109,0.4258,0.1704,0.6811,0.5107,-0.0005
Mens vs Womens,0.3454,0.6257,0.3454,0.0326,0.6583,0.6257,0.3407,0.0282,0.6531,0.6250,-0.0012
